In [16]:
import numpy as np
import pandas as pd

In [17]:
df = pd.read_csv('../data/isic2019/ISIC2019_Artifacts_merged.csv')

In [18]:
df.head()

,Unnamed: 0.1,image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,...,dark_corner,hair,gel_border,gel_bubble,ruler,ink,patches,vasc,label,label_string
0,0,ISIC_0000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.212077,0.082440,0.002752,0.006651,0.000267,0.000133,0.001456,0.001009,0,malignant
1,1,ISIC_0000001,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.008441,0.999747,0.000939,0.978680,0.003439,0.000301,0.000172,0.006230,0,malignant
2,2,ISIC_0000002,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.996333,0.098154,0.001869,0.014650,0.003663,0.001363,0.000764,0.039847,1,malignant
3,3,ISIC_0000003,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.005482,0.999364,0.000485,0.979981,0.000454,0.000594,0.000428,0.000076,0,malignant
4,4,ISIC_0000004,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.998930,0.928666,0.058840,0.009549,0.978695,0.000168,0.002042,0.034614,1,malignant


In [19]:
df.columns

Index(['Unnamed: 0.1', 'image', 'MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC',
       'SCC', 'UNK', 'Unnamed: 0', 'dark_corner', 'hair', 'gel_border',
       'gel_bubble', 'ruler', 'ink', 'patches', 'vasc', 'label',
       'label_string'],
      dtype='object')

In [20]:
#analyze the label distribution
print(df['label'].value_counts())

#analyze the disease distribution
diseases = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']
disease_counts = {disease: len(df[df[disease] == 1.0]) for disease in diseases}
print(disease_counts)

label
0    16858
1     3741
Name: count, dtype: int64
{'MEL': 3741, 'NV': 12875, 'BCC': 0, 'AK': 867, 'BKL': 2624, 'DF': 239, 'VASC': 253, 'SCC': 0, 'UNK': 0}


In [21]:
#create a subset of the dataframe with only MEL and NV
df_mel_nv = df[(df['MEL'] == 1.0) | (df['NV'] == 1.0)]
print(df_mel_nv['label'].value_counts())

label
0    12875
1     3741
Name: count, dtype: int64


In [22]:
#for this new dataset for the columns  ['dark_corner', 'hair', 'gel_border', 'gel_bubble', 'ruler', 'ink', 'patches'] put 0 if less than 0.8
#and 1 if greater than 0.8
artifacts = ['dark_corner', 'hair', 'gel_border', 'gel_bubble', 'ruler', 'ink', 'patches', 'vasc']
df_mel_nv = df_mel_nv.copy()

for artifact in artifacts:
    df_mel_nv[artifact] = df_mel_nv[artifact].apply(lambda x: 0 if x < 0.9 else 1)
#drop the column Unnamed: 0
df_mel_nv = df_mel_nv.drop(columns=['BCC', 'AK', 'BKL', 'DF', 'VASC',
       'SCC', 'UNK', 'Unnamed: 0.1', 'Unnamed: 0', 'label_string'])

#split the data into train, validation and test
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df_mel_nv, test_size=0.15, random_state=42)
df_train, df_val = train_test_split(df_train, test_size=0.15, random_state=42)

#save the dataframes
df_train.to_csv('../data/isic2019/isic2019_train.csv', index=False)
df_val.to_csv('../data/isic2019/isic2019_val.csv', index=False)
df_test.to_csv('../data/isic2019/isic2019_test.csv', index=False)

In [23]:
df_mel_nv.head()

,image,MEL,NV,dark_corner,hair,gel_border,gel_bubble,ruler,ink,patches,vasc,label
0,ISIC_0000000,0.0,1.0,0,0,0,0,0,0,0,0,0
1,ISIC_0000001,0.0,1.0,0,1,0,1,0,0,0,0,0
2,ISIC_0000002,1.0,0.0,1,0,0,0,0,0,0,0,1
3,ISIC_0000003,0.0,1.0,0,1,0,1,0,0,0,0,0
4,ISIC_0000004,1.0,0.0,1,1,0,0,1,0,0,0,1


# Visualize ISIC2019

In [44]:
from PIL import Image
import torch, torchvision
from torchvision import transforms

In [45]:
img = Image.open('/home/jupyter/ISIC_2019_Training_Input/orig/ISIC_0000023_downsampled.jpg').convert('RGB')

In [47]:
transforms = transforms.Compose([
                transforms.Resize((512, 512)),
                transforms.ToTensor(),
                transforms.Normalize(
                                mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225]
                            )
                ])
image = transforms(img)
torchvision.utils.save_image(image, f'test.jpg')
        